In [ ]:
import torch
import torch.nn as nn
import time
from torch.utils.data import DataLoader
import os
import pandas as pd

import utils


data_path = utils.check_cifar_dataset_exists()
utils.check_mnist_dataset_exists()

device = torch.device('cuda') 

In [39]:
import copy

def replace_multiple_conv_layers(base_model, generated_layers, positions, input_channels=1):
    encoding_copy = copy.deepcopy(base_model) 
    cnn_layers = encoding_copy[0]
    
    conv_count = 0
    gen_idx = 0
    prev_out_channels = input_channels

    for i in range(len(cnn_layers)):
        layer = cnn_layers[i]

        if len(layer) == 4:  # Conv2D layer
            if conv_count in positions:
                gen_layer = generated_layers[gen_idx]
                out_channels = int(gen_layer[0])
                kernel_size = int(gen_layer[1])
                padding = int(gen_layer[2])

                cnn_layers[i] = [prev_out_channels, out_channels, kernel_size, padding]
                gen_idx += 1
                if gen_idx >= len(generated_layers):
                    break

            prev_out_channels = layer[1]
            conv_count += 1

    return encoding_copy  


## LeNet model encoding

In [5]:
# Defined by data
input_channels = 1
output_dim=10

le_net_encoding = [
    [
        #Conv(input_channels,output_channels,kernel_size,padding), Maxpool(Kernel_size,Stride)
        [input_channels, 50, 3, 1], [2, 2], 
        [50, 100, 3, 1], [2, 2]
    ],
    [
        # MLP layers
        [4900, 100], 
        [100, output_dim]
    ]
]

### Child Network Class

In [24]:
# Child model definition and functions
class ChildCNNModel(nn.Module):
    def __init__(self, encoding, input_channels,height,width, output_dim):
        super(ChildCNNModel, self).__init__()
        self.input_channels = input_channels
        self.height = height
        self.width = width
        self.input_dim = self.input_channels * self.height * self.width
        self.output_dim = output_dim
        self.model = self.build_cnn_model_from_encoding(encoding)

    def forward(self, x):
        return self.model(x)



    def build_cnn_model_from_encoding(self,le_net_encoding):
        cnn_layers = []
        c, h, w = self.input_channels, self.height, self.width
        convCount=0
    
        for layer in le_net_encoding[0]:
            if len(layer) == 4:
                in_channels, out_channels, kernel_size, padding = layer
                if convCount==0:
                    in_channels = self.input_channels
                cnn_layers.append(nn.Conv2d(in_channels, out_channels, kernel_size, padding=padding))
                cnn_layers.append(nn.ReLU())
    
                h = (h + 2 * padding - kernel_size) + 1
                w = (w + 2 * padding - kernel_size) + 1
                c = out_channels
                convCount += 1
    
            elif len(layer) == 2:
                kernel_size, stride = layer
                cnn_layers.append(nn.MaxPool2d(kernel_size=kernel_size, stride=stride))
    
                h = (h - kernel_size) // stride + 1
                w = (w - kernel_size) // stride + 1
    
        flattened_dim = c * h * w
        cnn_layers.append(nn.Flatten())
    
        fc_layers = []
        fc_encoding = le_net_encoding[1]
    
        # Replace the first layer's in_features with computed flattened_dim
        fc_layers.append(nn.Linear(flattened_dim, fc_encoding[0][1]))
        fc_layers.append(nn.ReLU())
    
        for i in range(1, len(fc_encoding) - 1):
            in_features, out_features = fc_encoding[i]
            fc_layers.append(nn.Linear(in_features, out_features))
            fc_layers.append(nn.ReLU())
    
        # Final layer to output_dim
        last_in = fc_encoding[-1][0]
        fc_layers.append(nn.Linear(last_in, self.output_dim))
    
        return nn.Sequential(*cnn_layers, *fc_layers)

    def train_model(self, data, label, criterion, optimizer, device, epochs=10):
        self.model.train()  # Set the model to training mode
        total_loss = 0  # Total loss of the model
        start_time = time.time()  # Start time of the training
        bs = 200

        self.model.to(device)

        for _ in range(epochs):
            shuffled_indices = torch.randperm(data.size(0))
            num_batches = 0
            running_loss = 0

            for iter in range(1, len(data), bs):
                num_batches += 1

                # Set dL/dU, dL/dV, dL/dW to be filled with zeros
                optimizer.zero_grad()

                # create a minibatch
                indices = shuffled_indices[iter:iter + bs]
                minibatch_data = data[indices]
                minibatch_label = label[indices]

                # send batch to device
                minibatch_data = minibatch_data.to(device)
                minibatch_label = minibatch_label.to(device)

                # reshape the minibatch
                inputs = minibatch_data.view(-1, self.input_channels, self.height, self.width)

                # tell Pytorch to start tracking all operations that will be done on "inputs"
                inputs.requires_grad_()

                # forward the minibatch through the net
                scores = self.model(inputs)

                # Compute the average of the losses of the data points in the minibatch
                loss = criterion(scores, minibatch_label)
                running_loss += loss.detach().item()

                # backward pass to compute dL/dU, dL/dV and dL/dW
                loss.backward()

                # do one step of stochastic gradient descent: U=U-lr(dL/dU), V=V-lr(dL/dU), ...
                optimizer.step()

            total_loss = running_loss / num_batches

        elapsed_time = time.time() - start_time
        return total_loss, elapsed_time

    def evaluate_model(self, data, labels, device):
        self.model.eval()

        bs = 200
        correct = 0
        total = 0

        with torch.no_grad():
            for i in range(0, data.size(0), bs):
                # Slice the batch manually
                minibatch_data = data[i:i + bs].to(device)
                minibatch_labels = labels[i:i + bs].to(device)

                inputs = minibatch_data.view(-1, self.input_channels,self.height,self.width)

                # Forward pass
                scores = self.model(inputs)
                predicted = torch.argmax(scores, dim=1)

                # Count correct predictions
                total += minibatch_labels.size(0)
                correct += torch.sum(predicted == minibatch_labels).item()

        return correct / total

### RNN Controller

In [19]:
import torch
import torch.nn as nn

# Define search space
FILTER_CHOICES = [16, 32, 64, 128]
KERNEL_CHOICES = [1, 3, 5]
PADDING_CHOICES = [0, 1, 2]

FILTER_VOCAB = {str(val): idx for idx, val in enumerate(FILTER_CHOICES)}
KERNEL_VOCAB = {str(val): idx for idx, val in enumerate(KERNEL_CHOICES)}
PADDING_VOCAB = {str(val): idx for idx, val in enumerate(PADDING_CHOICES)}

IDX_TO_FILTER = {idx: val for val, idx in FILTER_VOCAB.items()}
IDX_TO_KERNEL = {idx: val for val, idx in KERNEL_VOCAB.items()}
IDX_TO_PADDING = {idx: val for val, idx in PADDING_VOCAB.items()}


class CNNController(nn.Module):
    def __init__(self, embedding_dim=8, hidden_dim=32, num_layers=1):
        super(CNNController, self).__init__()
        self.embedding = nn.Embedding(1, embedding_dim)  # Dummy embedding for input
        self.rnn = nn.LSTM(embedding_dim, hidden_dim, num_layers, batch_first=True)
        self.fc_filter = nn.Linear(hidden_dim, len(FILTER_CHOICES))
        self.fc_kernel = nn.Linear(hidden_dim, len(KERNEL_CHOICES))
        self.fc_padding = nn.Linear(hidden_dim, len(PADDING_CHOICES))

    def forward(self, x, hidden=None):
        x = self.embedding(x)
        output, hidden = self.rnn(x, hidden)
        filter_logits = self.fc_filter(output)
        kernel_logits = self.fc_kernel(output)
        padding_logits = self.fc_padding(output)
        return filter_logits, kernel_logits, padding_logits, hidden

    def generate_sequence(self, max_layers=1):
        self.eval()
        input_token = torch.zeros(1, 1, dtype=torch.long)  # Dummy start token
        hidden = None
        sequence = []
        log_probs = []

        for _ in range(max_layers):
            f_logits, k_logits, p_logits, hidden = self.forward(input_token, hidden)

            f_dist = torch.distributions.Categorical(logits=f_logits[:, -1, :])
            k_dist = torch.distributions.Categorical(logits=k_logits[:, -1, :])
            p_dist = torch.distributions.Categorical(logits=p_logits[:, -1, :])

            f_token = f_dist.sample()
            k_token = k_dist.sample()
            p_token = p_dist.sample()

            log_prob = f_dist.log_prob(f_token) + k_dist.log_prob(k_token) + p_dist.log_prob(p_token)

            layer = (IDX_TO_FILTER[f_token.item()], IDX_TO_KERNEL[k_token.item()], IDX_TO_PADDING[p_token.item()])
            sequence.append(layer)
            log_probs.append(log_prob)

        return sequence, torch.stack(log_probs)


# Initialize the model
model = CNNController()
print(model.generate_sequence())  # Example output


([('64', '5', '1')], tensor([[-3.6826]], grad_fn=<StackBackward0>))


### Model Generation and Training

In [32]:
test_controller = CNNController()

test_sequence, _ = test_controller.generate_sequence(max_layers=1)

print(test_sequence)

# Replacing 2nd conv layer with new layer
new_le_net_encoding = replace_multiple_conv_layers(base_model=le_net_encoding, generated_layers=test_sequence, positions=[1], input_channels=1)

print(f"Modified model encoding {new_le_net_encoding}")

train_dataset = torch.load(data_path + 'cifar/train_data.pt',weights_only=True)
train_labels = torch.load(data_path + 'cifar/train_label.pt',weights_only=True)

if len(train_dataset.size())==3:
    train_dataset = train_dataset.unsqueeze(1)

num_channels = train_dataset.size(1)
height = train_dataset.size(2)
width = train_dataset.size(3)



test_model = ChildCNNModel(new_le_net_encoding, num_channels, height, width, 10).to(device)
print(test_model)


test_model_loss, train_time = test_model.train_model(
    train_dataset,
    train_labels,
    nn.CrossEntropyLoss(),
    torch.optim.SGD(test_model.parameters(), lr=0.001),
    device,
    epochs=10
)
print("child model final epoch loss:", test_model_loss, "with train time:", train_time)

test_accuracy = test_model.evaluate_model(
    train_dataset,
    train_labels,
    device
)
print("child model validation set accuracy:", test_accuracy)

[('32', '3', '0')]
Modified model encoding [[[1, 50, 3, 1], [2, 2], [50, 32, 3, 0], [2, 2]], [[4900, 100], [100, 10]]]
ChildCNNModel(
  (model): Sequential(
    (0): Conv2d(1, 50, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): ReLU()
    (2): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (3): Conv2d(50, 32, kernel_size=(3, 3), stride=(1, 1))
    (4): ReLU()
    (5): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (6): Flatten(start_dim=1, end_dim=-1)
    (7): Linear(in_features=1152, out_features=100, bias=True)
    (8): ReLU()
    (9): Linear(in_features=100, out_features=10, bias=True)
  )
)
child model final epoch loss: 2.1153259921073913 with train time: 10.796806573867798
child model validation set accuracy: 0.5553


### RL Loop

In [33]:
def is_valid_encoding(encoding):
    if len(encoding) < 3 or encoding[0] != "START" or encoding[-1] != "END":
        return False

    for i in range(1, len(encoding) - 1):
        if not encoding[i].isnumeric():
            return False

    return True


def split_dataset(data, labels, split_ratio=0.8):
    dataset = torch.utils.data.TensorDataset(data, labels)
    train_size = int(split_ratio * len(dataset))
    test_size = len(dataset) - train_size

    train_set, test_set = torch.utils.data.random_split(dataset, [train_size, test_size])

    train_data, train_labels = zip(*train_set)
    train_data = torch.stack(train_data)
    train_labels = torch.stack(train_labels)

    test_data, test_labels = zip(*test_set)
    test_data = torch.stack(test_data)
    test_labels = torch.stack(test_labels)

    return (train_data, train_labels), (test_data, test_labels)

In [47]:
def train_controller(controller, train, test=None, model_iters=5, train_epochs=10, negative_reward=-1,
                     max_child_layers=5):
    if not test:
        train, test = split_dataset(*train)
    


    train_x, train_y = train

    test_x, test_y = test
    
    if len(train_x.size())==3:
        train_x = train_x.unsqueeze(1)
        
    if len(test_x.size())==3:
        test_x = test_x.unsqueeze(1)
        
    print(f"Training set shape: {train_x.size()}")

    num_channels = train_x.size(1)
    height = train_x.size(2)
    width = train_x.size(3)
    output_dim = torch.max(train_y).item() + 1

    optimizer = torch.optim.Adam(controller.parameters(), lr=0.001)
    gamma = 0.9

    baseline = 0

    for iter in range(model_iters):
        print(f"------ ITERATION {iter} ------")

        layer_encoding, log_probs = controller.generate_sequence(max_layers=max_child_layers)
        print("Model encoding:", layer_encoding)
        model_encoding  = replace_multiple_conv_layers(base_model=le_net_encoding, generated_layers=layer_encoding, positions=[1], input_channels=1)
        
        child_model = ChildCNNModel(model_encoding, num_channels,height,width, output_dim).to(device)

        print(child_model)

        # Train child model
        child_model_loss, train_time = child_model.train_model(
            train_x,
            train_y,
            nn.CrossEntropyLoss(),
            torch.optim.SGD(child_model.parameters(), lr=0.01),
            device,
            epochs=train_epochs
        )
        print("CHILD MODEL LOSS:", child_model_loss, "with train time:", train_time)

        accuracy = child_model.evaluate_model(test_x, test_y, device)
        print("CHILD MODEL ACCURACY:", accuracy)

        reward = accuracy

        advantage = torch.tensor(reward - baseline, dtype=torch.float32)
        # advantage = torch.clamp(advantage, -0.1, 0.1)

        policy_gradient = -torch.sum(log_probs) * advantage

        baseline = (1 - gamma) * reward + gamma * baseline

        optimizer.zero_grad()
        policy_gradient.backward()
        optimizer.step()

        print("Policy gradient:", policy_gradient.item())

        print()

# Is It Okay If Higher Accuracy → Higher Loss (policy gradient)?
#
# Yes, because you're minimizing negative reward-weighted log-probabilities:
#
# Higher reward = higher positive loss value = stronger update in backward()
# Lower reward = smaller or even negative advantage = smaller gradient or a push in the opposite direction
# Loss value increasing is not a problem — because you’re not minimizing “loss” in the usual supervised learning sense — you're maximizing expected reward through gradient ascent via REINFORCE.


In [49]:
train_data = (torch.load(data_path + 'cifar/train_data.pt',weights_only=True), torch.load(data_path + 'cifar/train_label.pt',weights_only=True))
test_data = (torch.load(data_path + 'cifar/test_data.pt',weights_only=True), torch.load(data_path + 'cifar/test_label.pt',weights_only=True))

train_controller(CNNController(), train_data, test_data, model_iters=20, train_epochs=50, negative_reward=-1,
                 max_child_layers=1)

C:\Users\vaibh\AppData\Local\Temp\ipykernel_17920\2927245077.py:1: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  train_data = (torch.load(data_path + 'cifar/train_data.pt'),

Training set shape: torch.Size([50000, 3, 32, 32])
------ ITERATION 0 ------
Model encoding: [('64', '1', '0'), ('64', '1', '2'), ('64', '3', '1'), ('128', '3', '0'), ('16', '3', '1')]
ChildCNNModel(
  (model): Sequential(
    (0): Conv2d(3, 50, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): ReLU()
    (2): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (3): Conv2d(50, 64, kernel_size=(1, 1), stride=(1, 1))
    (4): ReLU()
    (5): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (6): Flatten(start_dim=1, end_dim=-1)
    (7): Linear(in_features=4096, out_features=100, bias=True)
    (8): ReLU()
    (9): Linear(in_features=100, out_features=10, bias=True)
  )
)


KeyboardInterrupt: 

In [48]:
train_data = (torch.load(data_path + 'mnist/train_data.pt',weights_only=True), torch.load(data_path + 'mnist/train_label.pt',weights_only=True))
test_data = (torch.load(data_path + 'mnist/train_data.pt',weights_only=True), torch.load(data_path + 'mnist/train_label.pt',weights_only=True))

train_controller(CNNController(), train_data, test_data, model_iters=20, train_epochs=10, negative_reward=-1,
                 max_child_layers=1)

C:\Users\vaibh\AppData\Local\Temp\ipykernel_17920\730315531.py:1: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  train_data = (torch.load(data_path + 'mnist/train_data.pt'), 

Training set shape: torch.Size([60000, 1, 28, 28])
------ ITERATION 0 ------
Model encoding: [('64', '5', '1'), ('64', '3', '1'), ('64', '1', '1')]
ChildCNNModel(
  (model): Sequential(
    (0): Conv2d(1, 50, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): ReLU()
    (2): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (3): Conv2d(50, 64, kernel_size=(5, 5), stride=(1, 1), padding=(1, 1))
    (4): ReLU()
    (5): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (6): Flatten(start_dim=1, end_dim=-1)
    (7): Linear(in_features=2304, out_features=100, bias=True)
    (8): ReLU()
    (9): Linear(in_features=100, out_features=10, bias=True)
  )
)
CHILD MODEL LOSS: 0.12332723451157411 with train time: 14.154829263687134
CHILD MODEL ACCURACY: 0.9650166666666666
Policy gradient: 9.952077865600586

------ ITERATION 1 ------
Model encoding: [('32', '5', '0'), ('128', '5', '1'), ('32', '1', '0')]
ChildCNNModel(
  (model): Se

KeyboardInterrupt: 

In [14]:
# Simple functions to load datasets as data, labels pairs. Return type should be tensors

def load_music_dataset():
    if not os.path.exists('data/music_genre_classification.csv'):
        print("Download from Kaggle: https://www.kaggle.com/datasets/purumalgi/music-genre-classification")
        raise FileNotFoundError('data/music_genre_classification.csv')

    df = pd.read_csv('data/music_genre_classification.csv')
    df = df.dropna()

    # Artist embeddings
    artist_to_id = {artist: idx for idx, artist in enumerate(df['Artist Name'].unique())}
    df['artist_id'] = df['Artist Name'].map(artist_to_id)
    artist_embedding_layer = nn.Embedding(len(artist_to_id), 10)
    artist_embeddings = artist_embedding_layer(torch.LongTensor(df['artist_id'].values)).detach()

    # Track embeddings
    track_to_id = {track: idx for idx, track in enumerate(df['Track Name'].unique())}
    df['track_id'] = df['Track Name'].map(track_to_id)
    track_embedding_layer = nn.Embedding(len(track_to_id), 10)
    track_embeddings = track_embedding_layer(torch.LongTensor(df['track_id'].values)).detach()

    # Normalize numerical features using PyTorch
    numerical_cols = [
        'Popularity', 'danceability', 'energy', 'key', 'loudness', 'mode',
        'speechiness', 'acousticness', 'instrumentalness', 'liveness',
        'valence', 'tempo', 'duration_in min/ms', 'time_signature'
    ]
    x = torch.tensor(df[numerical_cols].values, dtype=torch.float32)
    mean = x.mean(dim=0, keepdim=True)
    std = x.std(dim=0, keepdim=True)
    std[std == 0] = 1.0  # Avoid divide-by-zero
    normalized_x = (x - mean) / std

    # Final dataset
    data = torch.hstack((artist_embeddings, track_embeddings, normalized_x))
    labels = torch.LongTensor(df['Class'].values)

    return data, labels


load_music_dataset()

Download from Kaggle: https://www.kaggle.com/datasets/purumalgi/music-genre-classification


FileNotFoundError: data/music_genre_classification.csv

In [ ]:
data, labels = load_music_dataset()
tx, ty = split_dataset(data, labels)
print(data.shape, labels.shape)
print(tx[0].shape, tx[1].shape, ty[0].shape, ty[1].shape)

In [ ]:
train_controller(CNNController(), (data, labels), model_iters=20, train_epochs=10, negative_reward=-1,
                 max_child_layers=1)